# 4 — A study, and what it says

Level 3: N training runs. It has **no type** — a study is a `for` — and that is
on purpose. A graph earns its keep when there are dependencies to declare, and
between two trials there are none.

What lives in the library is what the `for` asks for, and it all has the same
shape: **numbers in, a decision out, never a tensor**. That is what lets it all
be Rust while the loop stays in Python, where torch is. No callback crosses.

In [ ]:
import math
import tempfile

import torch

import soma_next.torch  # noqa: F401
from soma_next import Graph, Node, Opaque, Store, _theme
from soma_next.study import (
    DONE,
    PRUNED,
    Pruner,
    Sampler,
    Space,
    curves,
    finished,
    report,
    take,
    trials,
)
from soma_next.torch import Trainer, parameters

torch.manual_seed(0)

## The space is declared, and it is read back from text

A record keeps a point **as text** beside its score, so rebuilding a whole
history costs one scan and no fetches. Text alone is ambiguous — is `batch=64`
a number or an option called `"64"`? — so the space is what parses it.

In [ ]:
space = (
    Space()
    .real("lr", 1e-4, 1e-1, log=True)
    .int("width", 8, 64)
    .choice("opt", ["adam", "sgd"])
)
space

## `ask` is a function of the trial **number**

Not of what was asked before. That is the property the whole distributed half
rests on: a machine that claims trial 7 works out where to look on its own,
without replaying the first six and without asking anybody.

`Halton` and `Sobol` cover the space **for every prefix** rather than in
expectation, which is what stops two machines proposing neighbours.

In [ ]:
sampler = Sampler.sobol(seed=0)
for trial in range(4):
    print(trial, sampler.ask(space, trial, []))

## The loop

`take` claims a trial, `report` writes down where it got to, `finished` reads
the history back. They are **functions over a `Store`**, like `gather`: what is
being touched is a folder, and a class around one would be the store with a
longer name.

Run this same script on eight machines over a shared folder and it is a
distributed study. There is no server, no port and no protocol: a trial is a
number, and `claim` settles who gets it — so the state **is** the queue.

In [ ]:
truth = torch.randn(8, 1)


def batch(how_many=64):
    x = torch.randn(how_many, 8)
    return x, x @ truth + 0.1 * torch.randn(how_many, 1)


class Body(Node):
    def __init__(self, width):
        self.net = torch.nn.Sequential(torch.nn.Linear(8, width), torch.nn.ReLU())

    def forward(self, x, ctx):
        return Opaque(self.net(x))

    def parameters(self):
        return list(self.net.parameters())


class Head(Node):
    def __init__(self, width):
        self.out = torch.nn.Linear(width, 1)

    def forward(self, x, ctx):
        return Opaque(self.out(x))

    def parameters(self):
        return list(self.out.parameters())


def trained(point, epochs, pruner=None, so_far=()):
    """One trial: builds what the point says, trains it, reports as it goes."""
    g = Graph.somatize(Body(point["width"]).named("body") >> Head(point["width"]).named("head"))
    make = torch.optim.Adam if point["opt"] == "adam" else torch.optim.SGD
    t = Trainer(g, objective=torch.nn.functional.mse_loss,
                optimizer=make(parameters(g), lr=point["lr"]))
    said = []
    for epoch in range(epochs):
        said.append(sum(t.step(batch()) for _ in range(10)) / 10)
        # A pruner **stops nothing**: it answers, and the loop stops calling.
        if pruner is not None and (why := pruner.verdict(said, list(so_far))):
            return said, why
    return said, None

In [ ]:
STUDY = "widths"
store = Store(tempfile.mkdtemp())
me = "this machine"
pruner = Pruner.median(goal="min", warmup=4, startup=6)

for trial in range(30):
    point = sampler.ask(space, trial, finished(store, space, study=STUDY))
    if not take(store, point, study=STUDY, trial=trial, me=me):
        continue  # somebody else has that one
    # `curves` is the one reader that pays: a curve grows, so it lives in the
    # blob and this is a scan plus a fetch per trial. Everything else is a scan.
    said, why = trained(point, epochs=8, pruner=pruner, so_far=curves(store, study=STUDY))
    report(store, point, said, study=STUDY, trial=trial, me=me,
           state=PRUNED if why else DONE, because=why)

seen = trials(store, space, study=STUDY)
print(len(seen), "trials,", sum(one["state"] == PRUNED for one in seen), "pruned")

## The table of results

`trials` is one scan and no fetches — the configuration and the score are both
in the record, which is what makes a study readable from a machine that ran
none of it.

**A pruned score is not comparable with a finished one.** It is real, and it
was measured after fewer epochs, so ranking the two together says a trial that
was stopped early did badly when all that is known is that it was stopped. The
table shows both with their state; everything that follows uses only the ones
that ran to the end, and that is why `finished` leaves pruned trials out too.

In [ ]:
import plotly.graph_objects as go

scored = sorted((one for one in seen if one["score"] is not None), key=lambda one: one["score"])
done = [one for one in scored if one["state"] == DONE]
columns = ["trial", "state", "lr", "width", "opt", "score"]
rows = [
    [one["trial"] for one in scored],
    [one["state"] for one in scored],
    [f"{one['point']['lr']:.2e}" for one in scored],
    [one["point"]["width"] for one in scored],
    [one["point"]["opt"] for one in scored],
    [f"{one['score']:.4f}" for one in scored],
]

go.Figure(
    go.Table(
        header=dict(
            values=[f"<b>{c}</b>" for c in columns],
            fill_color=_theme.RAISED,
            line_color=_theme.EDGE,
            font=dict(color=_theme.INK, size=12),
            align="left",
            height=30,
        ),
        cells=dict(
            values=rows,
            fill_color=_theme.GROUND,
            line_color=_theme.EDGE,
            font=dict(color=_theme.INK, size=11),
            align="left",
            height=26,
        ),
    )
).update_layout(
    **_theme.layout(
        title=_theme.titled(f"{STUDY} — {len(scored)} scored, {len(done)} of them finished"),
        height=40 + 28 * (len(scored) + 1),
    )
)

## Which knob mattered

**Spearman's ρ**, which is a rank correlation: how well the score follows each
knob, monotonically, without assuming a shape. The original soma documents
fANOVA as deferred and never implemented it — what it has is this, and this is
about thirty lines of plain Python.

Ranks and not values, so a log-scaled knob needs no special case.

In [ ]:
def ranked(values):
    """Ranks, averaging ties — which is what makes it Spearman and not Pearson
    on whatever order the list happened to be in."""
    order = sorted(range(len(values)), key=lambda i: values[i])
    ranks = [0.0] * len(values)
    i = 0
    while i < len(order):
        j = i
        while j + 1 < len(order) and values[order[j + 1]] == values[order[i]]:
            j += 1
        shared = (i + j) / 2 + 1
        for k in range(i, j + 1):
            ranks[order[k]] = shared
        i = j + 1
    return ranks


def rho(xs, ys):
    """Spearman's ρ. `0` when a knob never varied: no evidence, not no effect."""
    a, b = ranked(xs), ranked(ys)
    n = len(a)
    mean_a, mean_b = sum(a) / n, sum(b) / n
    top = sum((x - mean_a) * (y - mean_b) for x, y in zip(a, b))
    below = math.sqrt(sum((x - mean_a) ** 2 for x in a) * sum((y - mean_b) ** 2 for y in b))
    return top / below if below else 0.0


def influence(scored, knobs):
    """|ρ| per knob against the score, biggest first. Categorical knobs are
    ranked by their own order, which is honest for two options and gets thin
    beyond that — a rank correlation over unordered categories is a number you
    should not lean on."""
    scores = [one["score"] for one in scored]
    said = {}
    for knob in knobs:
        values = [one["point"][knob] for one in scored]
        if isinstance(values[0], str):
            seen = sorted(set(values))
            values = [seen.index(v) for v in values]
        said[knob] = abs(rho(values, scores))
    return sorted(said.items(), key=lambda kv: -kv[1])


# Only the ones that ran to the end: see above.
mattered = influence(done, ["lr", "width", "opt"])
print(len(done), "finished trials ->", mattered)

In [ ]:
go.Figure(
    go.Bar(
        x=[value for _, value in reversed(mattered)],
        y=[knob for knob, _ in reversed(mattered)],
        orientation="h",
        marker=dict(color=_theme.SERIES["loss"], line=dict(color=_theme.EDGE, width=1)),
        text=[f"{value:.2f}" for _, value in reversed(mattered)],
        textposition="outside",
        textfont=dict(color=_theme.MUTED, size=11),
        cliponaxis=False,
    )
).update_layout(
    **_theme.layout(
        title=_theme.titled("|ρ| against the score — bigger is more decisive"),
        height=90 + 40 * len(mattered),
        showlegend=False,
        bargap=0.4,
    )
).update_xaxes(**_theme.axis(range=[0, 1])).update_yaxes(**_theme.axis(showgrid=False))

## Parallel coordinates

Every trial is a line across the knobs, coloured by its score. It is the one
picture that shows a *region* of the space rather than one knob at a time —
where the good lines bunch together is where to look next.

The axis for a log-scaled knob is drawn in log, which the original decides by
`max/min >= 50`.

In [ ]:
scores = [one["score"] for one in done]


def axis_for(knob):
    values = [one["point"][knob] for one in done]
    if isinstance(values[0], str):
        seen = sorted(set(values))
        return dict(label=knob, values=[seen.index(v) for v in values],
                    tickvals=list(range(len(seen))), ticktext=seen)
    spread = max(values) / min(values) if min(values) > 0 else 1
    if spread >= 50:
        return dict(label=f"{knob} (log)", values=[math.log10(v) for v in values])
    return dict(label=knob, values=values)


go.Figure(
    go.Parcoords(
        line=dict(
            color=scores,
            colorscale="Viridis",
            # Reversed, because lower is better here and the eye reads bright as
            # good. The original chooses this by the study's direction, and
            # getting it backwards is a figure that lies quietly.
            reversescale=True,
            showscale=True,
            colorbar=dict(
                title=dict(text="score", font=dict(color=_theme.MUTED, size=11)),
                tickfont=dict(color=_theme.MUTED, size=10),
                thickness=12,
            ),
        ),
        dimensions=[axis_for(k) for k in ("lr", "width", "opt")] + [dict(label="score", values=scores)],
        labelfont=dict(color=_theme.INK),
        tickfont=dict(color=_theme.MUTED),
        rangefont=dict(color=_theme.MUTED),
    )
).update_layout(
    **_theme.layout(
        title=_theme.titled(f"{STUDY} — every trial as a line"),
        height=420,
        # Room above: a parcoords writes its axis names along the top edge, and
        # the shared margin has no idea it is about to be sat on.
        margin={"l": 70, "r": 40, "t": 96, "b": 40},
    )
)

## What is not here

`curves(store, study=…)` is the reader a pruner uses, and it is the one that
**pays**: a curve grows, so it lives in the blob, and reading them is a scan
plus one fetch per trial. Everything above cost one scan.

And promoting these three figures into the library — a table, an influence
bar, parallel coordinates — is a slice of its own. They are here as an example
because they need nothing new to exist, which is exactly why they have not
been rushed into the API.